# 04.3 · Modelagem — regras de associação

**CRISP-DM fase 4 — modelagem** · Meridiano Imobiliária

---

Entrada: a base pré-processada do notebook [`03-preparacao`](03-preparacao.ipynb).

As regras de associação tratam cada imóvel como uma **cesta de características** (bairro, faixa de área, tem lareira…). Elas procuram combinações que aparecem juntas com frequência e as transformam em regras **se A, então B**.

| Algoritmo | Como encontra os conjuntos frequentes |
|---|---|
| **Apriori** | nível a nível: gera candidatos de tamanho *k* a partir dos frequentes de tamanho *k*−1 e conta cada um na base |
| **Eclat** | vertical: guarda, para cada item, a lista de imóveis que o têm, e conta interseções dessas listas em profundidade |
| **FP-Growth** | comprime a base numa árvore de prefixos (FP-tree) e minera a árvore, sem gerar candidatos |

Os três **encontram exatamente os mesmos conjuntos**; o que muda é o tempo. Por isso o notebook roda os três, confere que o resultado bate e só então gera as regras.

As métricas de cada regra A → B:
* **suporte:** fração dos imóveis com A e B juntos;
* **confiança:** entre os imóveis com A, a fração que também tem B;
* **lift:** quantas vezes A aumenta a chance de B. Lift 1 significa que A e B são independentes.

In [ ]:
import warnings
warnings.filterwarnings('ignore')  # silencia avisos de depreciação do mlxtend

import time
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from mlxtend.frequent_patterns import apriori, fpgrowth  # pacote mlxtend

MIN_SUPORTE = 0.15     # conjunto precisa aparecer em ao menos 15% dos imóveis
MAX_ITENS = 3          # tamanho máximo dos conjuntos (regras com até 3 itens)
SUPORTE_MAXIMO = 0.90  # itens em mais de 90% dos imóveis (rua pavimentada, gás…) não informam nada
MIN_CONFIANCA = 0.60   # regra precisa acertar em ao menos 60% dos casos
MIN_LIFT = 1.20        # e aumentar em ao menos 20% a chance do consequente
TOP = 15               # quantas regras aparecem nas tabelas

## 1 · Dos dados às transações

Cada coluna da saída do 03 vira um ou mais **itens**:
* **binárias** (one-hot e indicadores `tem_*`): viram item quando valem 1, como `zoneamento=RL` ou `tem_lareira`;
* **numéricas** (contínuas e ordinais escalonadas): são cortadas em **tercis** (baixo, médio, alto). O MinMax do 03 preserva a ordem dos valores, então os tercis são os mesmos da base original.

O `MonthlyCharges` entra como item, porque é bom candidato a consequente (*características → preço alto*). A `subclasse` sai, por ser um código.

In [ ]:
base = pd.read_csv('../data/processed/Telcon-Customer-Churn_preprocessed.csv', sep=',')  # saída do 03
base = base.drop(columns=['subclasse'], errors='ignore')                           # código do tipo de imóvel, não medida

df = pd.read_csv('../data/processed/Telcon-Customer-Churn_tratado.csv', sep=',')  # base curada: só os nomes das variáveis originais
variavel = {c: max((o for o in df.columns if c == o or c.startswith(f'{o}_')), key=len, default=c)
            for c in base.columns}                                       # cada feature → variável de origem (desfaz o one-hot)


def faixas(serie):
    # Corta uma coluna numérica em tercis; colunas com muitos valores repetidos ficam com 2 faixas.
    bordas = np.unique(serie.quantile([0, 1 / 3, 2 / 3, 1]))
    if len(bordas) < 3:                                                   # quase constante: não vira item
        return None
    rotulos = {2: ['baixo', 'alto'], 3: ['baixo', 'medio', 'alto']}[len(bordas) - 1]
    return pd.cut(serie, bordas, labels=rotulos, include_lowest=True)


itens = {}
for coluna in base.columns:
    valores = base[coluna]
    if set(valores.unique()) <= {0, 1}:                                   # binária: item quando vale 1
        categoria = coluna[len(variavel[coluna]) + 1:]                    # parte do nome depois da variável de origem
        itens[f'{variavel[coluna]}={categoria}' if categoria else coluna] = valores == 1
    elif (corte := faixas(valores)) is not None:                          # numérica: um item por faixa
        for faixa in corte.cat.categories:
            itens[f'{coluna}={faixa}'] = corte == faixa

transacoes = pd.DataFrame(itens)                                          # Clientes × serviço, True quando o cliente tem o serviço
suporte_item = transacoes.mean()                                          # fração de clientes com cada serviço
transacoes = transacoes.loc[:, suporte_item <= SUPORTE_MAXIMO]            # descarta itens quase universais
N = len(transacoes)

print(f'{N} Clientes · {len(itens)} itens gerados · {transacoes.shape[1]} mantidos '
      f'(descartados por estarem em mais de {SUPORTE_MAXIMO:.0%}: {sorted(suporte_item[suporte_item > SUPORTE_MAXIMO].index)})')
print(f'Itens com suporte ≥ {MIN_SUPORTE:.0%}: {(transacoes.mean() >= MIN_SUPORTE).sum()}')

## 2 · Os três algoritmos

O `mlxtend` traz Apriori e FP-Growth, mas não o Eclat. A implementação abaixo faz a busca em profundidade sobre as listas verticais, e a interseção de duas listas é um `&` entre vetores booleanos.

In [ ]:
def eclat(transacoes, min_suporte, max_itens):
    vertical = {item: transacoes[item].to_numpy() for item in transacoes.columns
                if transacoes[item].mean() >= min_suporte}               # item → em quais imóveis ele aparece
    encontrados = []

    def expandir(prefixo, presenca, candidatos):
        for i, item in enumerate(candidatos):
            juntos = presenca & vertical[item]                            # imóveis com o prefixo E o item
            suporte = juntos.mean()
            if suporte >= min_suporte:                                    # frequente: guarda e tenta crescer
                conjunto = prefixo + [item]
                encontrados.append((suporte, frozenset(conjunto)))
                if len(conjunto) < max_itens:
                    expandir(conjunto, juntos, candidatos[i + 1:])        # só itens à frente: evita repetir conjuntos

    expandir([], np.ones(len(transacoes), dtype=bool), sorted(vertical))
    return pd.DataFrame(encontrados, columns=['support', 'itemsets'])


algoritmos = {  # nome: função que devolve os conjuntos frequentes (colunas support e itemsets)
    'Apriori':   lambda: apriori(transacoes, min_support=MIN_SUPORTE, max_len=MAX_ITENS, use_colnames=True, low_memory=True),
    'Eclat':     lambda: eclat(transacoes, MIN_SUPORTE, MAX_ITENS),
    'FP-Growth': lambda: fpgrowth(transacoes, min_support=MIN_SUPORTE, max_len=MAX_ITENS, use_colnames=True),
}

resultados, tempos = {}, {}
for nome, rodar in algoritmos.items():
    inicio = time.perf_counter()
    resultados[nome] = rodar()                                            # conjuntos frequentes
    tempos[nome] = time.perf_counter() - inicio
    print(f'{nome}: {len(resultados[nome]):,} conjuntos em {tempos[nome]:.2f}s')

In [ ]:
# Os três precisam encontrar os mesmos conjuntos, com o mesmo suporte
referencia = resultados['FP-Growth'].set_index('itemsets')['support']
for nome, itemsets in resultados.items():
    atual = itemsets.set_index('itemsets')['support']
    assert set(atual.index) == set(referencia.index), f'{nome} encontrou conjuntos diferentes'
    assert np.allclose(atual.loc[referencia.index], referencia), f'{nome} calculou suportes diferentes'
print('Os três algoritmos encontraram os mesmos conjuntos frequentes.')

ax = pd.Series(tempos).sort_values().plot.barh(figsize=(7, 2.5), color='tab:blue')  # o que muda entre eles é o tempo
ax.set(title=f'Tempo para minerar {len(referencia):,} conjuntos'.replace(',', '.'), xlabel='segundos')
ax.grid(alpha=0.3, axis='x')
plt.show()

## 3 · Regras

As regras saem dos conjuntos frequentes, que são os mesmos para os três algoritmos. As fórmulas ficam explícitas na função.

In [ ]:
def gerar_regras(itemsets):
    suporte = dict(zip(itemsets['itemsets'], itemsets['support']))       # suporte de cada conjunto frequente
    regras = []
    for conjunto, sup in suporte.items():
        for tamanho in range(1, len(conjunto)):                          # toda divisão do conjunto em A → B
            for antecedente in map(frozenset, combinations(conjunto, tamanho)):
                consequente = conjunto - antecedente
                confianca = sup / suporte[antecedente]                   # P(B | A)
                regras.append({
                    'antecedente': ' + '.join(sorted(antecedente)),
                    'consequente': ' + '.join(sorted(consequente)),
                    'itens': len(conjunto),
                    'suporte': sup,                                      # P(A e B)
                    'confianca': confianca,
                    'lift': confianca / suporte[consequente],            # P(B | A) / P(B)
                })
    return pd.DataFrame(regras)


regras = gerar_regras(resultados['FP-Growth'])
fortes = (
    regras
    .query('confianca >= @MIN_CONFIANCA and lift >= @MIN_LIFT')          # só regras confiáveis e informativas
    .sort_values(['lift', 'confianca'], ascending=False)
    .reset_index(drop=True)
)
print(f'{len(regras):,} regras geradas · {len(fortes):,} com confiança ≥ {MIN_CONFIANCA:.0%} e lift ≥ {MIN_LIFT}'.replace(',', '.'))

formato = {'suporte': '{:.1%}', 'confianca': '{:.1%}', 'lift': '{:.2f}'}
fortes.head(TOP).style.format(formato).bar(subset=['lift'], color='#9ecae1')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pontos = ax.scatter(regras['suporte'], regras['confianca'], c=regras['lift'], s=8,
                    cmap='viridis', alpha=0.6)                            # cada ponto é uma regra
ax.axhline(MIN_CONFIANCA, color='gray', ls=':')                          # corte de confiança
ax.set(title='Regras · suporte × confiança (cor = lift)', xlabel='suporte', ylabel='confiança')
fig.colorbar(pontos, label='lift')
ax.grid(alpha=0.3)
plt.show()

## 4 · O que leva ao preço

As regras mais úteis para a Telcon são as que têm o **preço como consequente**: quais combinações de características indicam um imóvel caro ou barato.

In [ ]:
for faixa in ('alto', 'baixo'):
    selecao = (
        fortes[fortes['consequente'] == f'preco_venda={faixa}']          # regras que terminam numa faixa de preço
        .loc[lambda d: ~d['antecedente'].str.contains('preco_por_pe2')]  # preço por pé² contém o próprio preço: regra circular
        .head(10)
    )
    display(Markdown(f'**Características → preço {faixa}** ({len(selecao)} regras)'))
    display(selecao.style.format(formato).hide(axis='index'))

In [ ]:
fortes.to_csv('../data/processed/regras_associacao.csv', sep=',', index=False)  # regras fortes, para consulta
print(f'Salvas {len(fortes)} regras em ../data/processed/regras_associacao.csv')

## Como ler o que saiu

* **Apriori, Eclat e FP-Growth dão o mesmo resultado**, e a escolha entre eles é só de desempenho. Em bases pequenas como esta, a diferença é de segundos.
* **Regras óbvias aparecem no topo.** Itens derivados uns dos outros têm lift alto por construção, como `idade_imovel=alto` → `anos_desde_reforma=alto` ou `tem_garagem` → faixa de área da garagem. São verdadeiras, mas não ensinam nada.
* **Lift alto com suporte baixo é nicho.** A regra vale para poucos imóveis. Para decisões de carteira, prefira regras com suporte razoável.
* **Associação não é causa.** "Lareira → preço alto" não quer dizer que instalar uma lareira valoriza o imóvel. As duas coisas aparecem juntas em imóveis maiores e mais novos.
* **Para ajustar:** se surgirem regras demais, suba o `MIN_SUPORTE` ou o `MIN_LIFT`. Se surgirem de menos, desça o `MIN_SUPORTE` (o tempo cresce rápido) ou aumente o `MAX_ITENS`.